In [ ]:
# =============================================================================
# TASK 1: Multi-Agent Design Thinking
# =============================================================================
# Business Task: "Research a competitor, summarize findings, and draft a marketing angle"
#
# WHY 3 AGENTS?
# Each agent has a narrow focus. Specialist + specialist + specialist
# outperforms one generalist doing all three.
#
# | Agent        | Does                        | Does NOT              |
# |--------------|-----------------------------|-----------------------|
# | Researcher   | Gather raw data             | Interpret or analyze  |
# | Analyst      | Synthesize into insights    | Gather data or write  |
# | Strategist   | Draft marketing angle       | Research or analyze   |
# =============================================================================
#When a single agent is better: For simple tasks that don't require different expertise (e.g., "summarize this paragraph"), one agent is faster, cheaper, and just as good — multi-agent overhead only pays off when the task has distinct phases that benefit from different perspectives.

import os
from crewai import Agent, LLM


# --- ERROR HANDLING ---
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY not set. Run `export GROQ_API_KEY=your-key` before executing."
    )


# --- LLM CONFIG (one per agent, different temperatures) ---
# Researcher: temp=0 (factual, no randomness)
# Analyst: temp=0.3 (structured, slight flexibility)
# Strategist: temp=0.7 (creative, needs variety)
researcher_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.0)
analyst_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.3)
strategist_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.7)

# --- AGENT 1: RESEARCHER ---
researcher = Agent(
    role="Competitor Research Specialist",
    goal="Gather accurate, well-sourced factual information about the competitor — their products, pricing, market position, and recent activities — without adding opinion or interpretation.",
    backstory="You are a former competitive-intelligence analyst who spent years at a market research firm digging through product pages, press releases, and review sites. You are known for being exhaustive but strictly factual: you report what you find, cite where it came from, and never speculate about what it means for anyone else's strategy — that's not your job.",
    llm=researcher_llm,
    verbose=True,
    allow_delegation=False,
)

# --- AGENT 2: ANALYST ---
analyst = Agent(
    role="Competitive Insights Analyst",
    goal="Take raw competitor research and distill it into a tight, structured summary that surfaces the 3-5 findings that actually matter — strengths, weaknesses, gaps, and notable patterns — ready for someone else to act on.",
    backstory="You are a data-driven strategy analyst who has sat through too many meetings where raw research was dumped on stakeholders un-synthesized. Your specialty is pattern recognition: you take someone else's raw findings and turn them into a ranked, structured summary. You do not go looking for new facts, and you do not write marketing copy — you interpret what's already been gathered.",
    llm=analyst_llm,
    verbose=True,
    allow_delegation=False,
)

# --- AGENT 3: STRATEGIST ---
strategist = Agent(
    role="Marketing Angle Strategist",
    goal="Turn a structured competitive summary into a sharp, differentiated marketing angle and short pitch copy that our product can use — grounded strictly in the analyst's findings, not invented claims.",
    backstory="You are a scrappy positioning/copywriting specialist who has launched go-to-market campaigns for early-stage products. You're excellent at finding the one sentence that makes a product's edge obvious, but you've learned the hard way not to write copy before someone hands you real analysis — so you always build directly on the analyst's summary instead of researching or analyzing yourself.",
    llm=strategist_llm,
    verbose=True,
    allow_delegation=False,
)

# --- DESIGN VERIFICATION ---
print("=" * 60)
print("TASK 1: AGENT DESIGN")
print("=" * 60)
for a in (researcher, analyst, strategist):
    print(f"  - {a.role} | llm={a.llm.model} | temp={a.llm.temperature}")
print("=" * 60)

TASK 1: AGENT DESIGN (no tasks, no crew, no running)
  - Competitor Research Specialist | llm=groq/openai/gpt-oss-120b | temp=0.0
  - Competitive Insights Analyst | llm=groq/openai/gpt-oss-120b | temp=0.3
  - Marketing Angle Strategist | llm=groq/openai/gpt-oss-120b | temp=0.7


In [ ]:
# =============================================================================
# TASK 2: Build Agents & Assign Tools
# =============================================================================
# Builds on Task 1: adds tools to each agent
# Key rule: each agent gets only the tools they need for their role
# =============================================================================

import os
from crewai import Agent, LLM
from crewai.tools import tool

# --- LLM CONFIG (same as Task 1) ---
researcher_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.0)
analyst_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.3)
strategist_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.7)

# =============================================================================
# TOOLS — each agent gets only what they need
# =============================================================================

@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for information about a query. Returns key findings."""
    # In production, use real API (Google, Bing, SerpAPI)
    # For demo, return mock data
    return f"Search results for '{query}': Found competitor pricing, features, and market position data."

@tool("read_file")
def read_file(file_path: str) -> str:
    """Read and return contents of a file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"File not found: {file_path}"

@tool("write_file")
def write_file(file_path: str, content: str) -> str:
    """Write content to a file and confirm success."""
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Successfully wrote to {file_path}"
    except Exception as e:
        return f"Error writing file: {e}"

# =============================================================================
# AGENT 1: RESEARCHER
# Tools: web_search (find data), read_file (read sources)
# Justification: Researcher needs to GATHER information, not write it
# =============================================================================
researcher = Agent(
    role="Competitor Research Specialist",
    goal="Gather accurate, well-sourced factual information about the competitor.",
    backstory="You are a former competitive-intelligence analyst who is strictly factual. You report what you find and never speculate.",
    llm=researcher_llm,
    tools=[web_search, read_file],
    verbose=True,
    allow_delegation=False,
)

# =============================================================================
# AGENT 2: ANALYST
# Tools: write_file (save analysis)
# Justification: Analyst needs to SAVE structured output, it needs to read the researcher's saved output before it can analyze it — with only write_file it had no way to ingest the prior stage's work
# =============================================================================
analyst = Agent(
    role="Competitive Insights Analyst",
    goal="Take raw competitor research and distill it into a structured summary with strengths, weaknesses, and opportunities.",
    backstory="You are a data-driven analyst who turns raw information into strategic insights. You do not gather new data.",
    llm=analyst_llm,
    tools=[write_file, read_file],
    verbose=True,
    allow_delegation=False,
)

# =============================================================================
# AGENT 3: STRATEGIST
# Tools: write_file (save marketing angle)
# Justification: Strategist needs to SAVE creative output, it needs to read the analyst's summary before it can write copy grounded in it
# =============================================================================
strategist = Agent(
    role="Marketing Angle Strategist",
    goal="Turn a structured competitive summary into a sharp, differentiated marketing angle.",
    backstory="You are a copywriting specialist who builds directly on analysis. You never research or analyze yourself.",
    llm=strategist_llm,
    tools=[write_file, read_file],
    verbose=True,
    allow_delegation=False,
)

# =============================================================================
# VERIFICATION
# =============================================================================
print("=" * 60)
print("TASK 2: AGENTS WITH TOOLS")
print("=" * 60)
for a in (researcher, analyst, strategist):
    tool_names = [t.name for t in a.tools]
    print(f"  - {a.role}")
    print(f"    Tools: {tool_names}")
print("=" * 60)

TASK 2: AGENTS WITH TOOLS
  - Competitor Research Specialist
    Tools: ['web_search', 'read_file']
  - Competitive Insights Analyst
    Tools: ['write_file']
  - Marketing Angle Strategist
    Tools: ['write_file']


In [11]:
# =============================================================================
# WHY USE A .PY FILE INSTEAD OF RUNNING DIRECTLY IN JUPYTER?
# =============================================================================
# CrewAI v1.15.x has an async event loop conflict with Jupyter.
# crew.kickoff() fails in Jupyter because Jupyter already has an event loop running.
# Solution: Write code to a .py file and run it as a subprocess from Jupyter.
# This avoids the event loop conflict entirely.
# =============================================================================

# Run task3_runner.py as a subprocess
!python C:\Internship\Netixsol\week-2\day-4\task3_runner.py

RUNNING CREWAI CREW (SEQUENTIAL)
┌───────────────────────── 🚀 Crew Execution Started ─────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 5e4bf98a-d251-4a1f-8235-84d48518d09d                                   │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────── 📋 Task Started ──────────────────────────────┐
│                                                                             │
│  Task Started                                                               │
│  Name: Research Canva (the design tool company). Find: 1) Their main        │
│  produ

c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastapi\applications.py:18: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.exception_handlers import (
c:\Users\muham\AppData\Local\Programs\Python\Python312\Lib\site-packages\fastapi\applications.py:31: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  from fastapi.openapi.utils import get_openapi


In [ ]:
# =============================================================================
# WHY USE A .PY FILE?
# =============================================================================
# CrewAI v1.15.x has async event loop conflict with Jupyter.
# Writing to .py and running as subprocess avoids this entirely.
# =============================================================================

!python C:\Internship\Netixsol\week-2\day-4\task4_runner.py

In [ ]:
# =============================================================================
# WHY USE A .PY FILE?
# =============================================================================
# CrewAI v1.15.x has async event loop conflict with Jupyter.
# Writing to .py and running as subprocess avoids this entirely.
# =============================================================================

!python C:\Internship\Netixsol\week-2\day-4\task5_runner.py